In [0]:
%run ../../_shared/00_project_config

In [0]:
%run ../../_shared/01_spark_utils

In [0]:
from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Project-wide schema names are loaded from 00_project_config.
# Reusable validation, cleaning, deduplication, and writing functions are
# loaded from 01_spark_utils.

BRONZE_TABLE = f"{BRONZE_SCHEMA}.api_players_statistics"
SILVER_TABLE = f"{SILVER_SCHEMA}.players_statistics"

# One row in the Silver table represents a player's statistics for:
#   player + team + competition + season
BUSINESS_KEY = [
    "player_id",
    "team_id",
    "league_id",
    "league_season",
]

REQUIRED_BRONZE_COLUMNS = [
    "player",
    "statistics",
    "_ingested_at",
    "_source_file",
    "_source_system",
]

REQUIRED_SILVER_COLUMNS = [
    "player_id",
    "player_name",
    "team_id",
    "team_name",
    "league_id",
    "league_name",
    "league_season",
]


# -----------------------------------------------------------------------------
# Validate Bronze input
# -----------------------------------------------------------------------------
# A missing Bronze table is treated as a pipeline failure. Continuing without
# the required source would risk producing an incomplete or stale Silver table.

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise ValueError(
        f"Required Bronze table does not exist: {BRONZE_TABLE}"
    )


# -----------------------------------------------------------------------------
# Read Bronze data
# -----------------------------------------------------------------------------
# Bronze preserves the original API-Football structure:
#
#   player
#   statistics[]
#
# The statistics array contains team, league, games, goals, passes, cards,
# tackles, and other player-level season statistics.

bronze_df = spark.table(BRONZE_TABLE)

validate_columns_exist(
    bronze_df,
    REQUIRED_BRONZE_COLUMNS,
)


# -----------------------------------------------------------------------------
# Explode statistics array
# -----------------------------------------------------------------------------
# A single player object can contain multiple statistics records.
#
# Exploding the statistics array produces the intended Silver grain:
#
#   player + team + league + season
#
# Bronze lineage metadata is carried forward so duplicate source snapshots
# can later be resolved deterministically.

exploded_df = (
    bronze_df
    .select(
        F.col("player"),
        F.explode("statistics").alias("statistics"),
        F.col("_ingested_at")
        .cast("timestamp")
        .alias("_bronze_ingested_at"),
        F.col("_source_file")
        .cast("string")
        .alias("_bronze_source_file"),
        F.col("_source_system")
        .cast("string")
        .alias("_source_system"),
    )
)


# -----------------------------------------------------------------------------
# Select, flatten, clean, and cast Silver columns
# -----------------------------------------------------------------------------
# Nested API-Football structures are flattened into an analytical schema.
#
# clean_string() trims strings and converts empty strings to NULL.
#
# Source spelling mistakes such as:
#   appearences
#   commited
#
# are corrected only in the Silver column names. The Bronze layer remains
# faithful to the source structure.

silver_df = (
    exploded_df
    .select(

        # ---------------------------------------------------------------------
        # Player
        # ---------------------------------------------------------------------

        F.col("player.id")
        .cast("long")
        .alias("player_id"),

        clean_string("player.name")
        .alias("player_name"),

        clean_string("player.firstname")
        .alias("player_firstname"),

        clean_string("player.lastname")
        .alias("player_lastname"),

        F.col("player.age")
        .cast("int")
        .alias("player_age"),

        clean_string("player.nationality")
        .alias("player_nationality"),

        clean_string("player.height")
        .alias("player_height"),

        clean_string("player.weight")
        .alias("player_weight"),

        F.col("player.injured")
        .cast("boolean")
        .alias("player_injured"),

        F.to_date(
            F.col("player.birth.date")
        ).alias("player_birth_date"),

        clean_string("player.birth.place")
        .alias("player_birth_place"),

        clean_string("player.birth.country")
        .alias("player_birth_country"),


        # ---------------------------------------------------------------------
        # League / competition
        # ---------------------------------------------------------------------

        F.col("statistics.league.id")
        .cast("long")
        .alias("league_id"),

        clean_string("statistics.league.name")
        .alias("league_name"),

        clean_string("statistics.league.country")
        .alias("league_country"),

        F.col("statistics.league.season")
        .cast("int")
        .alias("league_season"),


        # ---------------------------------------------------------------------
        # Team
        # ---------------------------------------------------------------------

        F.col("statistics.team.id")
        .cast("long")
        .alias("team_id"),

        clean_string("statistics.team.name")
        .alias("team_name"),


        # ---------------------------------------------------------------------
        # Games
        # ---------------------------------------------------------------------

        F.col("statistics.games.appearences")
        .cast("int")
        .alias("games_appearances"),

        F.col("statistics.games.lineups")
        .cast("int")
        .alias("games_lineups"),

        F.col("statistics.games.minutes")
        .cast("int")
        .alias("games_minutes"),

        clean_string("statistics.games.position")
        .alias("games_position"),

        F.col("statistics.games.rating")
        .cast("double")
        .alias("games_rating"),

        F.col("statistics.games.captain")
        .cast("boolean")
        .alias("games_captain"),

        clean_string("statistics.games.number")
        .alias("games_number"),


        # ---------------------------------------------------------------------
        # Goals
        # ---------------------------------------------------------------------

        F.col("statistics.goals.total")
        .cast("int")
        .alias("goals_total"),

        F.col("statistics.goals.assists")
        .cast("int")
        .alias("goals_assists"),

        F.col("statistics.goals.conceded")
        .cast("int")
        .alias("goals_conceded"),

        F.col("statistics.goals.saves")
        .cast("int")
        .alias("goals_saves"),


        # ---------------------------------------------------------------------
        # Shots
        # ---------------------------------------------------------------------

        F.col("statistics.shots.total")
        .cast("int")
        .alias("shots_total"),

        F.col("statistics.shots.on")
        .cast("int")
        .alias("shots_on_target"),


        # ---------------------------------------------------------------------
        # Passes
        # ---------------------------------------------------------------------

        F.col("statistics.passes.total")
        .cast("int")
        .alias("passes_total"),

        F.col("statistics.passes.key")
        .cast("int")
        .alias("passes_key"),

        F.col("statistics.passes.accuracy")
        .cast("int")
        .alias("passes_accuracy"),


        # ---------------------------------------------------------------------
        # Tackles
        # ---------------------------------------------------------------------

        F.col("statistics.tackles.total")
        .cast("int")
        .alias("tackles_total"),

        F.col("statistics.tackles.blocks")
        .cast("int")
        .alias("tackles_blocks"),

        F.col("statistics.tackles.interceptions")
        .cast("int")
        .alias("tackles_interceptions"),


        # ---------------------------------------------------------------------
        # Duels
        # ---------------------------------------------------------------------

        F.col("statistics.duels.total")
        .cast("int")
        .alias("duels_total"),

        F.col("statistics.duels.won")
        .cast("int")
        .alias("duels_won"),


        # ---------------------------------------------------------------------
        # Dribbles
        # ---------------------------------------------------------------------

        F.col("statistics.dribbles.attempts")
        .cast("int")
        .alias("dribbles_attempts"),

        F.col("statistics.dribbles.success")
        .cast("int")
        .alias("dribbles_success"),

        clean_string("statistics.dribbles.past")
        .alias("dribbles_past"),


        # ---------------------------------------------------------------------
        # Fouls
        # ---------------------------------------------------------------------

        F.col("statistics.fouls.committed")
        .cast("int")
        .alias("fouls_committed"),

        F.col("statistics.fouls.drawn")
        .cast("int")
        .alias("fouls_drawn"),


        # ---------------------------------------------------------------------
        # Cards
        # ---------------------------------------------------------------------

        F.col("statistics.cards.yellow")
        .cast("int")
        .alias("cards_yellow"),

        F.col("statistics.cards.yellowred")
        .cast("int")
        .alias("cards_second_yellow"),

        F.col("statistics.cards.red")
        .cast("int")
        .alias("cards_red"),


        # ---------------------------------------------------------------------
        # Substitutions
        # ---------------------------------------------------------------------

        F.col("statistics.substitutes.bench")
        .cast("int")
        .alias("substitutes_bench"),

        F.col("statistics.substitutes.in")
        .cast("int")
        .alias("substitutes_in"),

        F.col("statistics.substitutes.out")
        .cast("int")
        .alias("substitutes_out"),


        # ---------------------------------------------------------------------
        # Penalties
        # ---------------------------------------------------------------------
        # API-Football exposes "commited" and "won" as source strings.
        # They remain strings for now because their exact source semantics
        # should be confirmed before converting them to another datatype.

        clean_string("statistics.penalty.commited")
        .alias("penalty_committed"),

        F.col("statistics.penalty.missed")
        .cast("int")
        .alias("penalty_missed"),

        F.col("statistics.penalty.saved")
        .cast("int")
        .alias("penalty_saved"),

        F.col("statistics.penalty.scored")
        .cast("int")
        .alias("penalty_scored"),

        clean_string("statistics.penalty.won")
        .alias("penalty_won"),


        # ---------------------------------------------------------------------
        # Bronze lineage metadata
        # ---------------------------------------------------------------------

        F.col("_bronze_ingested_at"),
        F.col("_bronze_source_file"),
        F.col("_source_system"),
    )
)


# -----------------------------------------------------------------------------
# Future entity standardization placeholders
# -----------------------------------------------------------------------------
# The next project phase will standardize names and identifiers across sources.
#
# Likely candidates from this dataset include:
#
#   player_name
#   player_firstname
#   player_lastname
#   player_nationality
#   player_birth_country
#   league_name
#   league_country
#   team_name
#
# The preferred future approach is to preserve the source values and introduce
# canonical platform identifiers / standardized values rather than overwriting
# source data.
#
# Example future model:
#
#   player_name_source
#   player_name_standardized
#   player_key
#
#   team_name_source
#   team_name_standardized
#   team_key
#
#   competition_key
#   country_key
#
# No physical placeholder columns are added yet because NULL-only or duplicate
# columns would increase schema noise before the standardization logic exists.


# -----------------------------------------------------------------------------
# Validate business key before deduplication
# -----------------------------------------------------------------------------
# Every Silver statistics record must be attributable to a specific:
#
#   player + team + league + season
#
# Validation happens before deduplication so NULL business keys cannot be
# grouped together and accidentally reduced to a single row.

validate_required_not_null(
    silver_df,
    BUSINESS_KEY,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Deduplicate repeated Bronze snapshots
# -----------------------------------------------------------------------------
# Bronze currently uses append mode. Re-running the Bronze load can therefore
# create multiple copies of the same player/team/league/season record.
#
# Keep the version from the latest Bronze ingestion event.

silver_df = deduplicate_latest(
    silver_df,
    key_columns=BUSINESS_KEY,
    order_column="_bronze_ingested_at",
    tie_breaker_columns=[
        "_bronze_source_file",
    ],
)


# -----------------------------------------------------------------------------
# Add Silver processing metadata
# -----------------------------------------------------------------------------

silver_df = silver_df.withColumn(
    "_silver_processed_at",
    F.current_timestamp(),
)


# -----------------------------------------------------------------------------
# Final Silver validation
# -----------------------------------------------------------------------------
# These columns are required for a usable player-statistics record.
#
# This validation also produces the final Silver row count without requiring
# an additional df.count() action.

validation_metrics = validate_required_not_null(
    silver_df,
    REQUIRED_SILVER_COLUMNS,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Write Silver table
# -----------------------------------------------------------------------------
# The current project phase uses full-refresh Silver processing.

write_full_refresh(
    silver_df,
    SILVER_TABLE,
)


# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {validation_metrics['row_count']} "
    f"player-statistics records into {SILVER_TABLE}."
)